In [ ]:
from pathlib import Path
import sys
import os
import datetime
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import cmocean.cm as cmo
current_dir = os.getcwd()
src_path = Path(current_dir).parent / "src"
sys.path.insert(0, str(src_path))
from fetch_heincke_data import heincke_download_underway_data
from fetch_voto_data import glider_download_nrt_data, sailbuoy_download_nrt_data

### Heincke data

We get this from the Heincke data API (You can see the details in `sr/fetch_heincke_data.py`)

By default, it fetches the most recent 24 hours. In this example, I ask for all the data from August. If you request too many rows the API will send an error! Pulling < 3 months is usually fine though. You can always request multiple batches, save them and combine them.

**N.B.** If you get an error trying to re-download, just load your existing data with the cell below

In [ ]:
df_heincke = heincke_download_underway_data(start=datetime.datetime(2025,8,15), end=datetime.datetime(2025,8,31))

In [ ]:
def read_heincke_data():
    df = pd.read_csv("heincke_raw_data.csv", sep='\t', parse_dates=['datetime'])

    df = df.rename({'vessel:heincke:trimble:longitude (mean) []': 'lon',
                    'vessel:heincke:trimble:latitude (mean) []': 'lat', 
                    'vessel:heincke:tsg:salinity (mean) [0/00]': 'salinity [PSU]',
                    'vessel:heincke:tsg:sbe38:temperature (mean) [°C]': 'temperature [°C]',
                    }, axis=1)
    return df
df_heincke = read_heincke_data()

In [ ]:
fig, ax = plt.subplots()
c = ax.scatter(df_heincke.lon, df_heincke.lat, c=df_heincke['temperature [°C]'], cmap=cmo.thermal)
ax.set(xlabel='longitude', ylabel='latitide', title='Heincke TSG data')
plt.colorbar(c, label='Temperature [°C]')

### Glider data

Here we are requesting an old glider mission, we'll update this with the correct mission once we have deployed the glider. You can pass any dataset id from the [VOTO ERDDAP](https://erddap.observations.voiceoftheocean.org/erddap/info/index.html)

In [ ]:
df_glider = glider_download_nrt_data(dataset_id="nrt_SEA044_M109")

In [ ]:
fig, ax = plt.subplots()

c = plt.scatter(df_glider.datetime, df_glider['depth (m)'], c=df_glider['temperature (Celsius)'], cmap=cmo.thermal)
ax.set(ylabel='depth', title='Glider data')
plt.colorbar(c, label='Temperature [°C]')
plt.xticks(rotation=45)
ax.invert_yaxis()


### Sailbuoy data

Very similar to glider data, just a different variable name for temperature (maybe we should standardise this upstream?)

In [ ]:
df_sailbuoy = sailbuoy_download_nrt_data()

In [ ]:
fig, ax = plt.subplots()
c = ax.scatter(df_sailbuoy.lon, df_sailbuoy.lat, c=df_sailbuoy['TEMP (degree_C)'], cmap=cmo.thermal)
ax.set(xlabel='longitude', ylabel='latitide', title='Sailbuoy data')
plt.colorbar(c, label='Temperature [°C]')